> ## SOLUTION / ANSWER KEY &mdash; Lab 6.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-external-apis.ipynb`](../lab-10-external-apis.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.10 &mdash; Connect to External APIs (Search + Wolfram)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap a Google-Search-style tool over a live fact source
- Wrap a Wolfram-Alpha-style exact-compute tool
- Bind them to an agent; optionally call the REAL APIs

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Connecting to real tools & APIs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-10"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Real agents reach the world through **tool integrations** (deck slide 16): **Google Search** for live
facts beyond the training cutoff, **Wolfram Alpha** for exact computation. The pattern is always: get a
key &rarr; wrap the service as a `@tool` &rarr; add it to the tools list. But **real APIs fail** &mdash;
rate limits, timeouts, bad queries &mdash; so a production tool must **wrap the call and return a safe
fallback** instead of crashing the whole agent. The **graded** cell wraps deterministic local stand-ins
(with the failure path) and binds them to a real agent; the **optional** cell calls the real APIs if you
have keys.

In [ ]:
# Real APIs fail: rate limits, timeouts, bad queries. A resilient tool must NOT crash the agent.
def flaky_source(query):
    """Stand-in for a real API that can fail -- raises when it has no data for the query."""
    index = {"gold price today usd per oz": "2400"}
    key = query.lower().strip()
    if key not in index:
        raise ConnectionError("upstream API error for: " + query)
    return index[key]

print("works :", flaky_source("gold price today usd per oz"))
try:
    flaky_source("who won the 3pm race")
except Exception as e:
    print("fails :", type(e).__name__, "-- a raw call would crash the agent")

## Your Turn
Make the **google_search** tool resilient: complete `safe_search` so a **failing** source returns the
`FALLBACK` instead of crashing. Then wrap it as the tool, add a **wolfram** compute tool, and bind both
to an agent with `create_agent`.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

FALLBACK = "no result (source unavailable)"   # the safe value to return when the source fails

def flaky_source(query):
    """Stand-in for a real API that can fail -- raises when it has no data for the query."""
    index = {"gold price today usd per oz": "2400", "eiffel tower height m": "330"}
    key = query.lower().strip()
    if key not in index:
        raise ConnectionError("upstream API error for: " + query)
    return index[key]

def safe_search(query):
    try:
        return flaky_source(query)
    except Exception:
        return FALLBACK

@tool
def google_search(query: str) -> str:
    """Search the web for a current fact or figure. Use for anything not in the model own memory."""
    return safe_search(query)   # resilient: wraps the flaky source

@tool
def wolfram(expression: str) -> float:
    """Compute an exact math expression (a Wolfram-Alpha-style compute tool)."""
    return safe_calc(expression)

def build_agent():
    model = ChatOllama(model="llama3.2:1b")
    return create_agent(model, [google_search, wolfram])

try:
    print('search success:', google_search.invoke('gold price today usd per oz'))
    print('search failure:', google_search.invoke('who won the 3pm race'))
    print('wolfram        :', wolfram.invoke('2400*2'))
    print('agent nodes    :', set(build_agent().nodes) - {'__start__'})
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("safe_search returns the live fact on the SUCCESS path", lambda: safe_search("gold price today usd per oz") == "2400")
expect_true("safe_search returns the FALLBACK on the FAILURE path (no crash)", lambda: safe_search("who won the 3pm race") == FALLBACK)
expect_true("the google_search tool is resilient too: unknown query -> fallback, not a crash", lambda: google_search.invoke("who won the 3pm race") == FALLBACK)
expect_true("the compute tool does exact math", lambda: wolfram.invoke("2400*2") == 4800)
expect_true("both external tools bind to the agent", lambda: type(build_agent()).__name__ == "CompiledStateGraph" and "tools" in set(build_agent().nodes))
expect_true("the search tool carries a real (filled-in) description", lambda: "___" not in google_search.description and "search" in google_search.description.lower())

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Swap the two stand-in tools for the REAL Google Serper and Wolfram Alpha wrappers. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
import os
try:
    from langchain_community.utilities import GoogleSerperAPIWrapper
    if os.getenv("SERPER_API_KEY"):
        print("Google (Serper):", GoogleSerperAPIWrapper().run("current gold price per ounce"))
    else:
        print("Set SERPER_API_KEY (serper.dev) to run a REAL Google search here -- skipping.")
except Exception as e:
    print("Install langchain-community for the real search wrapper -- skipping:", type(e).__name__)
try:
    from langchain_community.utilities.wolfram_alpha import WolframAlphaAPIWrapper
    if os.getenv("WOLFRAM_ALPHA_APPID"):
        print("Wolfram Alpha:", WolframAlphaAPIWrapper().run("2400 * 2"))
    else:
        print("Set WOLFRAM_ALPHA_APPID (developer.wolframalpha.com) for real compute -- skipping.")
except Exception as e:
    print("Install langchain-community + wolframalpha for the real wrapper -- skipping:", type(e).__name__)
print("The graded tools above already ran offline against a local stand-in.")

---
### Key takeaway
Get a key -> wrap the service as a @tool -> add it to the list. That is how an agent reaches live facts and real computation -- the client's Day-3 external-API lab, for real.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>